# Create Cost Management Ontology via Fabric REST API

This notebook programmatically creates an **Azure Cost Management Ontology** in Microsoft Fabric
using the Ontology REST API. It builds entity types, data bindings, relationship types, and
contextualizations, then deploys the ontology in a single API call.

**Prerequisites:**
- A Microsoft Fabric workspace with Contributor (or higher) permissions
- A Lakehouse containing 6 tables: `subscription`, `resource_group`, `resource`, `cost_record`, `meter_category`, `service`
- The Lakehouse must be attached to this notebook
- Run `01_download_cost_data.ipynb` first to populate the tables

**Entity Types:**
| Entity | Description | Key |
|--------|-------------|-----|
| Subscription | Azure subscription for billing | subscriptionId |
| ResourceGroup | Logical container for Azure resources | resourceGroupId |
| Resource | Individual Azure resource (VM, storage, etc.) | resourceId |
| CostRecord | Daily cost line item (fact) | costRecordId |
| MeterCategory | Billing meter classification (e.g. Virtual Machines, Storage) | meterId |
| Service | Consumed Azure service and charge type | serviceId |

**Relationships:**
- Subscription → has_resource_group → ResourceGroup
- ResourceGroup → contains_resource → Resource
- Resource → incurs_cost → CostRecord
- CostRecord → billed_under → MeterCategory
- CostRecord → consumed_by → Service
- Subscription → billed_by → CostRecord (direct subscription-level aggregation)


In [ ]:
# semantic-link (sempy.fabric) is pre-installed in Fabric runtimes — no pip install needed

In [ ]:
import sempy.fabric as fabric
import requests
import json
import base64
import uuid
import time

## 1. Parameters


In [ ]:
# ─── Load configuration ───────────────────────────────────────────────────────
# This notebook uses sempy.fabric (workspace identity) for Fabric API calls.
# No SPN credentials are needed — only the ontology display name.
import os

def load_env(filepath):
    """Parse a .env file and set os.environ variables."""
    if not os.path.exists(filepath):
        return False
    with open(filepath) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, value = line.partition("=")
            os.environ[key.strip()] = value.strip()
    return True

# Try .env file for optional config (ontology name, etc.)
for p in [
    os.path.join(os.path.dirname(os.getcwd()), ".env"),
    os.path.join(os.getcwd(), ".env"),
    "/lakehouse/default/Files/.env",
]:
    if load_env(p):
        print(f"Loaded .env from: {p}")
        break

# ─── User Parameters ───────────────────────────────────────────────────────────
workspace_name = None          # None = current workspace
lakehouse_name = None          # None = first lakehouse in workspace
ontology_display_name = os.environ.get("ONTOLOGY_DISPLAY_NAME", "CostManagementOntology")
ontology_description = "Azure cost breakdown: subscriptions, resource groups, resources, meters, and daily costs"

print(f"Ontology name: {ontology_display_name}")
print("Auth: Using Fabric workspace identity (no SPN credentials needed)")


## 2. Resolve Workspace and Lakehouse IDs


In [ ]:
workspace_id = fabric.resolve_workspace_id(workspace_name)
print(f"Workspace ID: {workspace_id}")

lakehouses_df = fabric.list_items(workspace=workspace_name, type="Lakehouse")
if lakehouse_name:
    lh_row = lakehouses_df[lakehouses_df["Display Name"] == lakehouse_name]
else:
    lh_row = lakehouses_df.head(1)

if lh_row.empty:
    raise ValueError("Lakehouse not found. Check lakehouse_name or attach a Lakehouse.")

lakehouse_id = str(lh_row.iloc[0]["Id"])
lakehouse_display = lh_row.iloc[0]["Display Name"]
print(f"Lakehouse: {lakehouse_display} ({lakehouse_id})")

# ─── Pre-flight: check which tables exist and their columns ──────────────────
available_tables = {}
for t in ["subscription", "resource_group", "resource", "cost_record", "meter_category", "service"]:
    try:
        tbl = spark.table(t)
        available_tables[t] = tbl.columns
        print(f"  ✓ {t}: {tbl.columns}")
    except Exception:
        print(f"  ✗ {t}: NOT FOUND — will be excluded from ontology")

if "cost_record" not in available_tables:
    raise RuntimeError("cost_record table is required. Run 01_download_cost_data.ipynb first.")


## 3. Helper Functions


In [ ]:
def to_b64(obj):
    """Convert a Python dict to a base64-encoded JSON string."""
    return base64.b64encode(json.dumps(obj, indent=2).encode("utf-8")).decode("utf-8")

_id_counter = 2000000000000
def next_id():
    global _id_counter
    _id_counter += 1
    return str(_id_counter)

# Track entity type IDs and property IDs
et_ids = {}
prop_ids = {}  # "Entity.property" -> id

def make_property(entity, name, value_type="String"):
    pid = next_id()
    prop_ids[f"{entity}.{name}"] = pid
    return {
        "id": pid,
        "name": name,
        "redefines": None,
        "baseTypeNamespaceType": None,
        "valueType": value_type,
    }

## 4. Define Entity Types

Each entity type maps to a Lakehouse table. Properties are defined with their Fabric value types.

### Field Meanings (from Azure Cost Management documentation):
- **subscriptionId / subscriptionName**: The Azure subscription being billed
- **resourceGroupId / resourceGroupName**: Logical container grouping related Azure resources
- **resourceId**: Full Azure Resource Manager path identifying a specific resource
- **resourceType**: The Azure resource provider type (e.g. Microsoft.Compute/virtualMachines)
- **location**: Azure region where the resource is deployed (e.g. eastus, westeurope)
- **meterCategory**: Top-level billing classification (e.g. Virtual Machines, Storage, Networking)
- **meterSubCategory**: Finer billing classification within a category (e.g. D-Series, General Block Blob)
- **meterName**: Specific meter name (e.g. D4s v3, LRS Data Stored)
- **unitOfMeasure**: Unit for quantity billing (e.g. 1 Hour, 1 GB/Month)
- **preTaxCost**: Cost before tax and credits in billing currency
- **quantity**: Number of units consumed
- **chargeType**: Whether the charge is Usage, Purchase, or Refund
- **serviceName (ConsumedService)**: The Azure service that emitted the usage (e.g. Microsoft.Compute)


In [ ]:
entity_definitions = {}

def table_has_col(table_name, col_name):
    """Check if a column exists in an available table."""
    return table_name in available_tables and col_name in available_tables[table_name]

# ─── Subscription ─────────────────────────────────────────────────────────────
if "subscription" in available_tables:
    et_ids["Subscription"] = next_id()
    sub_props = [
        make_property("Subscription", "subscriptionId", "String"),
    ]
    if table_has_col("subscription", "subscriptionName"):
        sub_props.append(make_property("Subscription", "subscriptionName", "String"))
    entity_definitions["Subscription"] = {
        "id": et_ids["Subscription"],
        "namespace": "usertypes",
        "baseEntityTypeId": None,
        "name": "Subscription",
        "entityIdParts": [prop_ids["Subscription.subscriptionId"]],
        "displayNamePropertyId": prop_ids.get("Subscription.subscriptionName", prop_ids["Subscription.subscriptionId"]),
        "namespaceType": "Custom",
        "visibility": "Visible",
        "properties": sub_props,
        "timeseriesProperties": [],
    }

# ─── ResourceGroup ────────────────────────────────────────────────────────────
if "resource_group" in available_tables:
    et_ids["ResourceGroup"] = next_id()
    rg_props = [
        make_property("ResourceGroup", "resourceGroupId", "String"),
    ]
    if table_has_col("resource_group", "resourceGroupName"):
        rg_props.append(make_property("ResourceGroup", "resourceGroupName", "String"))
    if table_has_col("resource_group", "subscriptionId"):
        rg_props.append(make_property("ResourceGroup", "subscriptionId", "String"))
    entity_definitions["ResourceGroup"] = {
        "id": et_ids["ResourceGroup"],
        "namespace": "usertypes",
        "baseEntityTypeId": None,
        "name": "ResourceGroup",
        "entityIdParts": [prop_ids["ResourceGroup.resourceGroupId"]],
        "displayNamePropertyId": prop_ids.get("ResourceGroup.resourceGroupName", prop_ids["ResourceGroup.resourceGroupId"]),
        "namespaceType": "Custom",
        "visibility": "Visible",
        "properties": rg_props,
        "timeseriesProperties": [],
    }

# ─── Resource ─────────────────────────────────────────────────────────────────
if "resource" in available_tables:
    et_ids["Resource"] = next_id()
    res_props = [make_property("Resource", "resourceId", "String")]
    for col in ["resourceType", "resourceGroupName", "resourceGroupId", "location"]:
        if table_has_col("resource", col):
            res_props.append(make_property("Resource", col, "String"))
    entity_definitions["Resource"] = {
        "id": et_ids["Resource"],
        "namespace": "usertypes",
        "baseEntityTypeId": None,
        "name": "Resource",
        "entityIdParts": [prop_ids["Resource.resourceId"]],
        "displayNamePropertyId": prop_ids["Resource.resourceId"],
        "namespaceType": "Custom",
        "visibility": "Visible",
        "properties": res_props,
        "timeseriesProperties": [],
    }

# ─── CostRecord ───────────────────────────────────────────────────────────────
if "cost_record" in available_tables:
    et_ids["CostRecord"] = next_id()
    cost_props = [make_property("CostRecord", "costRecordId", "BigInt")]
    cost_record_cols = available_tables["cost_record"]
    for col, vtype in [("subscriptionId", "String"), ("resourceGroupId", "String"),
                       ("resourceId", "String"), ("meterId", "String"), ("serviceId", "String"),
                       ("usageDate", "DateTime"), ("preTaxCost", "Double"), ("quantity", "Double"),
                       ("unitOfMeasure", "String"), ("chargeType", "String"),
                       ("location", "String"), ("Currency", "String")]:
        if col in cost_record_cols:
            cost_props.append(make_property("CostRecord", col, vtype))

    entity_definitions["CostRecord"] = {
        "id": et_ids["CostRecord"],
        "namespace": "usertypes",
        "baseEntityTypeId": None,
        "name": "CostRecord",
        "entityIdParts": [prop_ids["CostRecord.costRecordId"]],
        "displayNamePropertyId": prop_ids["CostRecord.costRecordId"],
        "namespaceType": "Custom",
        "visibility": "Visible",
        "properties": cost_props,
        "timeseriesProperties": [],
    }

# ─── MeterCategory ────────────────────────────────────────────────────────────
if "meter_category" in available_tables:
    et_ids["MeterCategory"] = next_id()
    meter_props = [make_property("MeterCategory", "meterId", "String")]
    for col in ["meterCategory", "meterSubCategory", "meterName", "unitOfMeasure"]:
        if table_has_col("meter_category", col):
            meter_props.append(make_property("MeterCategory", col, "String"))
    entity_definitions["MeterCategory"] = {
        "id": et_ids["MeterCategory"],
        "namespace": "usertypes",
        "baseEntityTypeId": None,
        "name": "MeterCategory",
        "entityIdParts": [prop_ids["MeterCategory.meterId"]],
        "displayNamePropertyId": prop_ids.get("MeterCategory.meterCategory", prop_ids["MeterCategory.meterId"]),
        "namespaceType": "Custom",
        "visibility": "Visible",
        "properties": meter_props,
        "timeseriesProperties": [],
    }

# ─── Service ──────────────────────────────────────────────────────────────────
if "service" in available_tables:
    et_ids["Service"] = next_id()
    svc_props = [make_property("Service", "serviceId", "String")]
    for col in ["serviceName", "chargeType"]:
        if table_has_col("service", col):
            svc_props.append(make_property("Service", col, "String"))
    entity_definitions["Service"] = {
        "id": et_ids["Service"],
        "namespace": "usertypes",
        "baseEntityTypeId": None,
        "name": "Service",
        "entityIdParts": [prop_ids["Service.serviceId"]],
        "displayNamePropertyId": prop_ids.get("Service.serviceName", prop_ids["Service.serviceId"]),
        "namespaceType": "Custom",
        "visibility": "Visible",
        "properties": svc_props,
        "timeseriesProperties": [],
    }

print("Entity types defined:")
for name, defn in entity_definitions.items():
    print(f"  {name}: {len(defn['properties'])} properties, key={defn['entityIdParts']}")
print(f"\nSkipped entities (table not found): {[t for t in ['Subscription','ResourceGroup','Resource','CostRecord','MeterCategory','Service'] if t not in entity_definitions]}")

In [ ]:
# Data binding map: entity -> (table, [(col_name, prop_key), ...])
# Only bind columns that exist in both the table and the entity definition
binding_specs = {
    "Subscription":  ("subscription",   ["subscriptionId", "subscriptionName"]),
    "ResourceGroup": ("resource_group",  ["resourceGroupId", "resourceGroupName", "subscriptionId"]),
    "Resource":      ("resource",        ["resourceId", "resourceType", "resourceGroupName", "resourceGroupId", "location"]),
    "CostRecord":    ("cost_record",     ["costRecordId", "subscriptionId", "resourceGroupId", "resourceId",
                                          "meterId", "serviceId", "usageDate", "preTaxCost", "quantity",
                                          "unitOfMeasure", "chargeType", "location", "Currency"]),
    "MeterCategory": ("meter_category",  ["meterId", "meterCategory", "meterSubCategory", "meterName", "unitOfMeasure"]),
    "Service":       ("service",         ["serviceId", "serviceName", "chargeType"]),
}

data_bindings = {}

for entity_name, (table_name, candidate_cols) in binding_specs.items():
    if entity_name not in entity_definitions:
        print(f"  Skipping {entity_name} — entity not defined")
        continue
    if table_name not in available_tables:
        print(f"  Skipping {entity_name} — table {table_name} not found")
        continue

    table_cols = available_tables[table_name]
    binding_id = str(uuid.uuid4())
    property_bindings = []

    for col_name in candidate_cols:
        prop_key = f"{entity_name}.{col_name}"
        if col_name in table_cols and prop_key in prop_ids:
            property_bindings.append({
                "sourceColumnName": col_name,
                "targetPropertyId": prop_ids[prop_key],
            })

    if not property_bindings:
        print(f"  Skipping {entity_name} — no matching columns")
        continue

    data_bindings[entity_name] = {
        "id": binding_id,
        "dataBindingConfiguration": {
            "dataBindingType": "NonTimeSeries",
            "propertyBindings": property_bindings,
            "sourceTableProperties": {
                "sourceType": "LakehouseTable",
                "workspaceId": workspace_id,
                "itemId": lakehouse_id,
                "sourceTableName": table_name,
                "sourceSchema": "dbo",
            }
        }
    }

print("Data bindings built:")
for name, db in data_bindings.items():
    cfg = db["dataBindingConfiguration"]
    print(f"  {name} -> {cfg['sourceTableProperties']['sourceTableName']} ({len(cfg['propertyBindings'])} columns)")

## 6. Build Relationship Types and Contextualizations

Six relationships connect the entity types, forming a graph that enables multi-hop queries
like: *Subscription → ResourceGroup → Resource → CostRecord → MeterCategory*


In [ ]:
# Only create relationships where both source and target entities exist
# and the bridge table has the required join columns
all_relationship_specs = [
    ("has_resource_group", "Subscription", "ResourceGroup", "resource_group",
     "subscriptionId", "Subscription.subscriptionId",
     "resourceGroupId", "ResourceGroup.resourceGroupId"),

    ("contains_resource", "ResourceGroup", "Resource", "resource",
     "resourceGroupId", "ResourceGroup.resourceGroupId",
     "resourceId", "Resource.resourceId"),

    ("incurs_cost", "Resource", "CostRecord", "cost_record",
     "resourceId", "Resource.resourceId",
     "costRecordId", "CostRecord.costRecordId"),

    ("billed_under", "CostRecord", "MeterCategory", "cost_record",
     "costRecordId", "CostRecord.costRecordId",
     "meterId", "MeterCategory.meterId"),

    ("consumed_by", "CostRecord", "Service", "cost_record",
     "costRecordId", "CostRecord.costRecordId",
     "serviceId", "Service.serviceId"),

    ("billed_by", "Subscription", "CostRecord", "cost_record",
     "subscriptionId", "Subscription.subscriptionId",
     "costRecordId", "CostRecord.costRecordId"),
]

relationship_definitions = []
contextualization_definitions = []

for (rel_name, src_entity, tgt_entity, bridge_table,
     src_col, src_prop_key, tgt_col, tgt_prop_key) in all_relationship_specs:

    # Skip if either entity doesn't exist
    if src_entity not in et_ids or tgt_entity not in et_ids:
        print(f"  Skipping {rel_name} — {src_entity} or {tgt_entity} not defined")
        continue

    # Skip if bridge table doesn't have the required columns
    if bridge_table not in available_tables:
        print(f"  Skipping {rel_name} — bridge table {bridge_table} not found")
        continue

    bridge_cols = available_tables[bridge_table]
    if src_col not in bridge_cols or tgt_col not in bridge_cols:
        print(f"  Skipping {rel_name} — bridge table missing {src_col} or {tgt_col}")
        continue

    # Skip if property IDs don't exist
    if src_prop_key not in prop_ids or tgt_prop_key not in prop_ids:
        print(f"  Skipping {rel_name} — property {src_prop_key} or {tgt_prop_key} not defined")
        continue

    rel_id = next_id()

    rel_def = {
        "namespace": "usertypes",
        "id": rel_id,
        "name": rel_name,
        "namespaceType": "Custom",
        "source": {"entityTypeId": et_ids[src_entity]},
        "target": {"entityTypeId": et_ids[tgt_entity]},
    }
    relationship_definitions.append((rel_id, rel_def))

    ctx_id = str(uuid.uuid4())
    ctx_def = {
        "id": ctx_id,
        "dataBindingTable": {
            "sourceType": "LakehouseTable",
            "workspaceId": workspace_id,
            "itemId": lakehouse_id,
            "sourceTableName": bridge_table,
            "sourceSchema": "dbo",
        },
        "sourceKeyRefBindings": [
            {"sourceColumnName": src_col, "targetPropertyId": prop_ids[src_prop_key]}
        ],
        "targetKeyRefBindings": [
            {"sourceColumnName": tgt_col, "targetPropertyId": prop_ids[tgt_prop_key]}
        ],
    }
    contextualization_definitions.append((rel_id, ctx_id, ctx_def))

print("Relationships defined:")
for rel_id, rel_def in relationship_definitions:
    print(f"  {rel_def['name']}: {rel_def['source']['entityTypeId']} -> {rel_def['target']['entityTypeId']}")


## 7. Assemble the Full Definition Payload


In [ ]:
parts = []

# .platform
platform = {
    "metadata": {
        "type": "Ontology",
        "displayName": ontology_display_name,
    }
}
parts.append({"path": ".platform", "payload": to_b64(platform), "payloadType": "InlineBase64"})

# definition.json
parts.append({"path": "definition.json", "payload": to_b64({}), "payloadType": "InlineBase64"})

# Entity Types + Data Bindings (only those that exist)
for entity_name in ["Subscription", "ResourceGroup", "Resource", "CostRecord", "MeterCategory", "Service"]:
    if entity_name not in entity_definitions or entity_name not in data_bindings:
        continue
    et_id = et_ids[entity_name]
    et_def = entity_definitions[entity_name]
    db_def = data_bindings[entity_name]

    parts.append({
        "path": f"EntityTypes/{et_id}/definition.json",
        "payload": to_b64(et_def),
        "payloadType": "InlineBase64",
    })
    parts.append({
        "path": f"EntityTypes/{et_id}/DataBindings/{db_def['id']}.json",
        "payload": to_b64(db_def),
        "payloadType": "InlineBase64",
    })

# Relationship Types + Contextualizations
for rel_id, rel_def in relationship_definitions:
    parts.append({
        "path": f"RelationshipTypes/{rel_id}/definition.json",
        "payload": to_b64(rel_def),
        "payloadType": "InlineBase64",
    })

for rel_id, ctx_id, ctx_def in contextualization_definitions:
    parts.append({
        "path": f"RelationshipTypes/{rel_id}/Contextualizations/{ctx_id}.json",
        "payload": to_b64(ctx_def),
        "payloadType": "InlineBase64",
    })

print(f"Total definition parts: {len(parts)}")
for p in parts:
    print(f"  {p['path']}")

## 7b. Debug — Dump Raw Payload to File

Writes the full API request body (with base64 payloads decoded) to the Lakehouse Files folder
so you can download and inspect it. Also writes a version with the raw base64 payloads intact.


In [ ]:
# ─── Build the request body ──────────────────────────────────────────────────
debug_body = {
    "displayName": ontology_display_name,
    "description": ontology_description,
    "definition": {"parts": parts},
}

# ─── Save raw payload (base64-encoded, exactly as sent to the API) ───────────
raw_path = "/lakehouse/default/Files/debug_ontology_payload_raw.json"
with open(raw_path, "w") as f:
    json.dump(debug_body, f, indent=2)
print(f"Raw payload written to: {raw_path}")
print(f"  Size: {os.path.getsize(raw_path):,} bytes")

# ─── Save decoded payload (base64 decoded to readable JSON) ──────────────────
import copy
decoded_body = copy.deepcopy(debug_body)
for part in decoded_body["definition"]["parts"]:
    try:
        decoded = json.loads(base64.b64decode(part["payload"]).decode("utf-8"))
        part["payload_decoded"] = decoded
        del part["payload"]
    except Exception:
        part["payload_decoded"] = "(could not decode)"

decoded_path = "/lakehouse/default/Files/debug_ontology_payload_decoded.json"
with open(decoded_path, "w") as f:
    json.dump(decoded_body, f, indent=2)
print(f"Decoded payload written to: {decoded_path}")
print(f"  Size: {os.path.getsize(decoded_path):,} bytes")

print("\n→ Download both files from Lakehouse → Files folder in the Fabric portal")

## 8. Create or Update the Ontology via REST API

If an ontology with the same name already exists, its **definition is updated in-place**
using the `updateDefinition` API. This preserves the ontology ID so that Data Agent
references remain valid across refreshes. Only if no matching ontology exists will a
new one be created.

In [ ]:
from notebookutils import mssparkutils

access_token = mssparkutils.credentials.getToken("https://api.fabric.microsoft.com")

url = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/ontologies"
headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json",
}

# ─── Check if ontology already exists ─────────────────────────────────────────
existing_ontology_id = None
print(f"Checking for existing ontology '{ontology_display_name}'...")
existing = requests.get(url, headers=headers)
if existing.status_code == 200:
    for item in existing.json().get("value", []):
        if item.get("displayName") == ontology_display_name:
            existing_ontology_id = item["id"]
            print(f"  Found existing ontology: {existing_ontology_id}")
            break
    else:
        print("  No existing ontology found — will create new.")

# ─── Update existing OR Create new ───────────────────────────────────────────
definition_payload = {"definition": {"parts": parts}}

if existing_ontology_id:
    # UPDATE in-place — preserves the ontology ID so Data Agent references stay valid
    update_url = f"{url}/{existing_ontology_id}/updateDefinition?updateMetadata=True"
    print(f"\nUpdating ontology definition in-place (ID preserved)...")
    print(f"POST {update_url}")
    print(f"Payload: {len(json.dumps(definition_payload)):,} bytes\n")

    response = requests.post(update_url, headers=headers, json=definition_payload)
else:
    # CREATE new ontology
    body = {
        "displayName": ontology_display_name,
        "description": ontology_description,
        "definition": {"parts": parts},
    }
    print(f"\nCreating new ontology '{ontology_display_name}'...")
    print(f"POST {url}")
    print(f"Payload: {len(json.dumps(body)):,} bytes\n")

    response = requests.post(url, headers=headers, json=body)

# ─── Handle response ─────────────────────────────────────────────────────────
if response.status_code == 200:
    print("Ontology definition updated successfully!")
    print(f"  ID: {existing_ontology_id} (unchanged — Data Agent references preserved)")

elif response.status_code == 201:
    result = response.json()
    print("Ontology created successfully!")
    print(f"  ID:           {result.get('id')}")
    print(f"  Display Name: {result.get('displayName')}")
    print(f"  Workspace:    {result.get('workspaceId')}")

elif response.status_code == 202:
    operation_id = response.headers.get("x-ms-operation-id")
    retry_after = int(response.headers.get("Retry-After", 30))
    action = "Update" if existing_ontology_id else "Create"
    print(f"Async operation started: {operation_id}")
    print(f"Polling every {retry_after}s...\n")

    operation_url = f"https://api.fabric.microsoft.com/v1/operations/{operation_id}"
    for attempt in range(30):
        time.sleep(retry_after)
        poll = requests.get(operation_url, headers=headers)
        if poll.status_code == 200:
            status = poll.json().get("status", "")
            print(f"  Attempt {attempt+1}: {status}")
            if status.lower() in ("succeeded", "completed"):
                if existing_ontology_id:
                    print(f"\nOntology updated successfully!")
                    print(f"  ID: {existing_ontology_id} (unchanged — Data Agent references preserved)")
                else:
                    result_url = f"{operation_url}/result"
                    res = requests.get(result_url, headers=headers)
                    if res.status_code == 200:
                        print(f"\nOntology created successfully!")
                        print(json.dumps(res.json(), indent=2))
                    else:
                        print(f"\nOperation completed. Check workspace for '{ontology_display_name}'.")
                break
            elif status.lower() == "failed":
                print(f"\n{action} operation FAILED.")
                print(json.dumps(poll.json(), indent=2))
                try:
                    result_url = f"{operation_url}/result"
                    err_res = requests.get(result_url, headers=headers)
                    print(f"\nOperation result ({err_res.status_code}):")
                    print(err_res.text[:2000])
                except Exception as e:
                    print(f"Could not fetch error details: {e}")
                break
    else:
        print("\nTimed out. Check Fabric portal.")
else:
    print(f"ERROR {response.status_code}")
    print(f"Response headers: {dict(response.headers)}")
    try:
        print(json.dumps(response.json(), indent=2))
    except Exception:
        print(response.text)

## 9. Verify


In [ ]:
items_df = fabric.list_items(workspace=workspace_name, type="Ontology")
display(items_df)


## Summary

| Component | Count | Details |
|-----------|-------|---------|
| Entity Types | 6 | Subscription, ResourceGroup, Resource, CostRecord, MeterCategory, Service |
| Data Bindings | 6 | One per entity, bound to Lakehouse tables |
| Relationship Types | 6 | has_resource_group, contains_resource, incurs_cost, billed_under, consumed_by, billed_by |
| Contextualizations | 6 | One per relationship, using bridge tables |

**Next steps:**
- Open the ontology in the Fabric portal to explore the visual graph canvas
- Run GQL queries in the graph editor
- Create a Data Agent with the ontology as a data source
